# Lesson 10 - AI Agents in Production

ဒီသင်ခန်းစာမှာ Microsoft Agent Framework (Python) ကို အသုံးပြုပြီး AI agents အတွက် **ထုတ်လုပ်မှု ပုံစံများ** ကို သင်ယူပါမယ်။ အဓိကအချက်များမှာ:

- **အမြင်ပြတင်နိုင်ခြင်း** — agent တွေရဲ့ အပြန်အလှန် ဆက်သွယ်မှုတွင် အချိန်နှင့် မှတ်တမ်းပြုလုပ်ခြင်း ထည့်သွင်းခြင်း
- **အကဲဖြတ်ခြင်း** — အဖြေ အရည်အသွေးကို အဆင့်သတ်မှတ်ရန် evaluator agent ကို အသုံးပြုခြင်း
- **ကုန်ကျစရိတ် စီမံခန့်ခွဲမှု** — token optimization နှင့် model ရွေးချယ်မှု အတွက် နည်းလမ်းများ

ဤအခြေအနေမှာ သွားလာစီမံခန့်ခွဲရေး အေးဂျင့်တစ်ခုဖြစ်ပြီး အသုံးပြုသူများကို ခရီးစဉ် စီစဉ်ရာတွင် ကူညီပေး၊ ထိန်းကြပ်မှုနှင့် အကဲဖြတ်မှုများကို မျက်နှာပြင်ပေါ်တွင် ထည့်သွင်းထားသည်။


## တပ်ဆင်ခြင်း


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ထုတ်လုပ်မှုပေါ်တွင် စဉ်းစားရန်အချက်များ

AI အေးဂျင့်များကို စမ်းသပ်မှုမှ ထုတ်လုပ်မှုအဆင့်သို့ ရွှေ့ပြောင်းရာတွင် အထူးဂရုပြုရမည့် အချက်သုံးချက်ရှိသည်-

1. **မြင်သာမှု** — အေးဂျင့်သည် ဘာလုပ်ဆောင်နေသည်၊ ကြာချိန် ဘယ်လောက်ယူနေသည်၊ ဘယ်ကိရိယာများကို ခေါ်နေသည်ကို မြင်ရဖို့လို အလုံးစုံ စရိုက်ကြည့်ရမည်။ ပုံစံပြုပြင်မှုနှင့် မှတ်တမ်းတင်မှုမရှိပါက ထုတ်လုပ်မှု အခက်အခဲများကို ဖြေရှင်းရန် မဖြစ်နိုင်ပါ။

2. **အကဲဖြတ်ခြင်း** — အလိုအလျောက် အရည်အသွေးစစ်ဆေးမှုများက အေးဂျင့်၏ တုံ့ပြန်ချက်များသည် တိကျမှန်ကန်ပြီး ပြည့်စုံနှင့် အထောက်အကူ ဖြစ်နေသည်ကို အချိန်အတောအတွင်း သေချာစေသည်။ အကဲဖြတ်သူအေးဂျင့်တစ်ယောက်က သတ်မှတ်ချက်များနှင့် ကိုက်ညီမှုအရ ဖြေချက်များကို မျှဝေကိုင်တွယ်နိုင်သည်။

3. **ကုန်ကျစရိတ် စီမံခန့်ခွဲမှု** — တိုးကင်အသုံးပြုမှုသည် ကုန်ကျစရိတ်တောတွင် တိုက်ရိုက်သက်ရောက်သည်။ Prompt အပြုအသွေး အာရုံစူးစိုက်မှု၊ မော်ဒယ်ရွေးချယ်မှုနှင့် Cache သုံးမှုက စျေးနှုန်းများကို ထိန်းချုပ်ပေးကာ အရည်အသွေး ထိခိုက်မှု မရှိစေရန် ကူညီပေးသည်။


## Building an Observable Agent

ကျွန်ုပ်တို့သည် ခရီးသွားကိရိယာများကို သတ်မှတ်ပြီး အေဂျင့် ခေါ်ဆိုမှုကို အချိန်တိုင်းတာခြင်းဖြင့် ထုပ်ပိုးထားကာ နောက်ကျမှုကို စောင့်ကြည့်နိုင်သည်။ ထုတ်လုပ်မှုတွင် OpenTelemetry သို့မဟုတ် ဆင်ဆာချိန်နည်း တူညီသည့် ပြန်ကြားရေး အောက်ခံနည်းပညာနှင့် ပေါင်းစည်းအသုံးပြုနိုင်သည်။


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## အကဲဖြတ်မှု ပုံစံများ

အများဆုံး ထုတ်လုပ်မှု ပုံစံတစ်ခုမှာ ဒုတိယ အေးဂျင့်ကို **အကဲဖြတ်သူ** အနေဖြင့် အသုံးပြုတာ ဖြစ်ပါတယ်။ အဲဒီ အကဲဖြတ်သူက သတ်မှတ်ထားတဲ့ စံနှုန်းတွေဖြစ်တဲ့ ပြည့်စုံမှု၊ တိကျမှု၊ အသုံးဝင်မှု စတဲ့ အချက်တွေကို အခြေခံပြီး ပရိုတိုက် အေးဂျင့်ရဲ့ တုံ့ပြန်ချက်ကို အမှတ်ပေးပါတယ်။

ဒါကအောက်ပါအရာတွေကို အထောက်အကူဖြစ်စေပါတယ်-
- အသုံးပြုသူတွေ ရောက်ခင် အသင့်သုံး မြင့်မားတဲ့ အရည်အသွေး အတားအဆီးများကို စနစ်တကျ စစ်ဆေးနိုင်ခြင်း
- ပရောမ့်များ သို့မဟုတ် မော်ဒယ်များ ပြောင်းလဲတဲ့အခါ ပြန်လည်ဆန်းစစ်မှု တွေကို ဖော်ထုတ်နိုင်ခြင်း
- အချိန်ကြာမြင့်စဉ် အေးဂျင့် စွမ်းဆောင်ရည်ကို မျှတစွာ စောင့်ကြပ်သိရှိခြင်း


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## ကုန်ကျစရိတ် စီမံခန့်ခွဲမှု မဟာဗျူဟာများ

ထုတ်လုပ်မှု AI အေးဂျင့်များအတွက် ကုန်ကျစရိတ် ထိန်းချုပ်ခြင်းမှာ အရေးကြီးပါသည်။ ၎င်းအတွက် အဓိက မဟာဗျူဟာများမှာ အောက်ပါအတိုင်းဖြစ်သည်-

| မဟာဗျူဟာ | ဖော်ပြချက် |
|---|---|
| **ထိပ်တန်းပြင်ဆင်မှု** | စနစ်ညွှန်ကြားမှုများကို တိုတောင်း ရှင်းလင်းစေပါ။ ထပ်ခါထပ်ခါဖြစ်သော အကြောင်းအရာများကို ဖယ်ရှား၍ input token များ လျော့ကျစေပါ။ |
| **မော်ဒယ် ရွေးချယ်မှု** | ရိုးရှင်းသည့် တာဝန်များ (ဥပမာ: အမျိုးအစားခွဲခြားခြင်း သို့မဟုတ် အချက်အလက်ထုတ်ယူခြင်း) အတွက် ပိုသေးငယ်ပြီး သက်သာသော မော်ဒယ်များ (ဥပမာ GPT-4o-mini) ကို အသုံးပြု၍ ရှုပ်ထွေးသော အတွေးအခေါ်များအတွက် ပိုကြီးသော မော်ဒယ်များ သီးသန့်ထားပါ။ |
| **ကက်ရှ်ဖွင့်ခြင်း** | ကိရိယာရလဒ်များနှင့် မကြာခဏ မေးခွန်းများကို ကက်ရှ်ဖွင့်၍ မလိုသည့် API ခေါ်ဆိုမှုများ ရှောင်ကြဉ်ပါ။ |
| **Token ဘတ်ဂျက်များ** | မမျှော်လင့်ထားသော ကြာရှည်သော တုံ့ပြန်ချက်များ မဖြစ်ပေါ်စေရန် `max_tokens` ကန့်သတ်ချက်များ သတ်မှတ်ပါ။ |
| **အစုအဝေးဖြင့် ဆောင်ရွက်ခြင်း** | တစ်ခါတည်း API ခေါ်ဆိုမှုတစ်ခုအတွင်း မျိုးစုံသော အသုံးပြုသူ မေးခွန်းများကို ထည့်သွင်း စုစည်းပါ။ |

လက်တွေ့တွင် အဆင့်လိုက် နည်းလမ်းကောင်းစနစ်ဖြစ်သည်- ရိုးရှင်းလွယ်ကူသော တောင်းဆိုချက်များကို အရှိန်မြန်ပြီး သက်သာသော မော်ဒယ်သို့ ဦးတည်ပို့ဆောင်ပြီး ရှုပ်ထွေးသော မေးခွန်းများသာ ပိုများစွာ အရာရှိသော မော်ဒယ်သို့ အဆင့်မြှင့်ပေးပါ။


## အကျဉ်းချုပ်

ဤသင်ခန်းစာတွင် သင် သိရှိခဲ့သောအကြောင်းအရာများမှာ -

၁။ **အေဂျင့်နှင့်ဆက်ဆံမှုများအတွက် ကြာချိန်နှင့် မှတ်တမ်းတင်ခြင်းဖြင့် တွေ့မြင်ယူခြင်းကို ထည့်သွင်းခြင်း၊ ထောက်လှမ်းခြင်းနှင့် စောင့်ကြည့်ခြင်းအခြေခံကို တည်ဆောက်ခြင်း။  
၂။ **အေဂျင့်၏ပြန်ကြားချက်များကို အပြည့်အ၀၊ တိကျမှုနှင့် အကူအညီရစေရေးအတွက် အကဲဖြတ်သူအေဂျင့်ဖြင့် အလိုအလျောက် အကဲဖြတ်ခြင်း။  
၃။ **စျေးကြေးများကို လျှော့ချရန် ပရုံ့များ မြှင့်တင်ခြင်း၊ မော်ဒယ် ရွေးချယ်ခြင်း၊ သိမ်းဆည်းမှုနှင့် token ဘတ်ဂျက်များ အသုံးပြုခြင်း။  

ဤထုတ်လုပ်မှုအဆင့်များသည် သင့် AI အေဂျင့်များကို ယုံကြည်ရကြောင်း၊ တိုင်းတာနိုင်ကြောင်းနှင့် စျေးနှုန်းထိရောက်စွာ လည်ပတ်နိုင်စေရန် ကူညီပေးပါသည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
